# Objectives of this lab

In this lab, we will:
- Walk through how a Chainlink oracle works
- Participate in our own oracle contract that collects values from the class
- Demonstrate resilience against adversarial values

In [ ]:
from web3 import Web3

infura_key = ''

# Change this to use your own RPC URL for Sepolia Testnet
web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))

print("Connected to Sepolia Testnet:", web3)

# Exploring production oracles (Chainlink):

[Chainlink](https://chain.link/) is one of the most popular oracle providers on Ethereum. Let's take a high-level look at how the smart contracts for [the ETH-USD oracle on Chainlink](https://data.chain.link/ethereum/mainnet/crypto-usd/eth-usd) work for an example of how oracles can be implemented.

An end-to-end view of a price feed such as this consists of three types of contracts: 
1. A consumer contract
2. A proxy contract
3. An aggregator contract

### Consumer contract

The consumer contract is any piece of code that queries the aggregator for the oracle values. For the assignments, we would be writing a contract in Solidity. Below is a sample code for doing the same in Python.

In [ ]:
# ETH/USD Chainlink data feed address (Sepolia)
ETH_USD_PRICE_FEED_ADDR = '0x694AA1769357215DE4FAC081bf1f309aDC325306'

# AggregatorV3Interface ABI
abi = '[{"inputs":[{"internalType":"address","name":"_aggregator","type":"address"},{"internalType":"address","name":"_accessController","type":"address"}],"stateMutability":"nonpayable","type":"constructor"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"int256","name":"current","type":"int256"},{"indexed":true,"internalType":"uint256","name":"roundId","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"updatedAt","type":"uint256"}],"name":"AnswerUpdated","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"uint256","name":"roundId","type":"uint256"},{"indexed":true,"internalType":"address","name":"startedBy","type":"address"},{"indexed":false,"internalType":"uint256","name":"startedAt","type":"uint256"}],"name":"NewRound","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"}],"name":"OwnershipTransferRequested","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"}],"name":"OwnershipTransferred","type":"event"},{"inputs":[],"name":"acceptOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"accessController","outputs":[{"internalType":"contract AccessControllerInterface","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"aggregator","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"_aggregator","type":"address"}],"name":"confirmAggregator","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"description","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_roundId","type":"uint256"}],"name":"getAnswer","outputs":[{"internalType":"int256","name":"","type":"int256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint80","name":"_roundId","type":"uint80"}],"name":"getRoundData","outputs":[{"internalType":"uint80","name":"roundId","type":"uint80"},{"internalType":"int256","name":"answer","type":"int256"},{"internalType":"uint256","name":"startedAt","type":"uint256"},{"internalType":"uint256","name":"updatedAt","type":"uint256"},{"internalType":"uint80","name":"answeredInRound","type":"uint80"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_roundId","type":"uint256"}],"name":"getTimestamp","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"latestAnswer","outputs":[{"internalType":"int256","name":"","type":"int256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"latestRound","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"latestRoundData","outputs":[{"internalType":"uint80","name":"roundId","type":"uint80"},{"internalType":"int256","name":"answer","type":"int256"},{"internalType":"uint256","name":"startedAt","type":"uint256"},{"internalType":"uint256","name":"updatedAt","type":"uint256"},{"internalType":"uint80","name":"answeredInRound","type":"uint80"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"latestTimestamp","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint16","name":"","type":"uint16"}],"name":"phaseAggregators","outputs":[{"internalType":"contract AggregatorV2V3Interface","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"phaseId","outputs":[{"internalType":"uint16","name":"","type":"uint16"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"_aggregator","type":"address"}],"name":"proposeAggregator","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"proposedAggregator","outputs":[{"internalType":"contract AggregatorV2V3Interface","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint80","name":"_roundId","type":"uint80"}],"name":"proposedGetRoundData","outputs":[{"internalType":"uint80","name":"roundId","type":"uint80"},{"internalType":"int256","name":"answer","type":"int256"},{"internalType":"uint256","name":"startedAt","type":"uint256"},{"internalType":"uint256","name":"updatedAt","type":"uint256"},{"internalType":"uint80","name":"answeredInRound","type":"uint80"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"proposedLatestRoundData","outputs":[{"internalType":"uint80","name":"roundId","type":"uint80"},{"internalType":"int256","name":"answer","type":"int256"},{"internalType":"uint256","name":"startedAt","type":"uint256"},{"internalType":"uint256","name":"updatedAt","type":"uint256"},{"internalType":"uint80","name":"answeredInRound","type":"uint80"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"_accessController","type":"address"}],"name":"setController","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"_to","type":"address"}],"name":"transferOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"version","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"}]'
# Set up contract instance
contract = web3.eth.contract(address=ETH_USD_PRICE_FEED_ADDR, abi=abi)

# Make call to latestRoundData()
latest_round_data = contract.functions.latestRoundData().call() # returned format is roundId, answer, startedAt, updatedAt, answeredInRound

price = latest_round_data[1] / 10**8 # reported with 8 decimals
print(f"Current oracle reported ETH-USD price: {price}")

### Proxy contract

Proxy contracts are on-chain proxies that point to the aggregator for a particular data feed. Using proxies enables the underlying aggregator to be upgraded without any service interruption to consuming contracts.

The proxy contract for ETH-USD is deployed [here on mainnet](https://etherscan.io/address/0x5f4ec3df9cbd43714fe2740f5e3616155c5b8419#code) and [here on sepolia](https://sepolia.etherscan.io/address/0x694AA1769357215DE4FAC081bf1f309aDC325306).

### Aggregator contract

The contract code for the aggregator is given [here](https://github.com/smartcontractkit/libocr/blob/master/contract/AccessControlledOffchainAggregator.sol). An aggregator is the contract that receives periodic data updates from the network of nodes that have agreed to post their data to a particular Chainlink feed. The data collection and aggregation is done off-chain and the data is then posted on-chain. Aggregators store aggregated data on-chain so that consumers can retrieve it and and act upon it within the same transaction.

You can access this data using the Data Feed address and the [AggregatorV3Interface contract](https://github.com/smartcontractkit/chainlink/blob/b5883718b03b99d46c67ee3a1fe2b3abf3e74385/contracts/src/v0.6/interfaces/AggregatorV3Interface.sol).

Aggregators receive updates from the oracle network only when the *Deviation Threshold* or *Heartbeat Threshold* triggers an update during an aggregation round. The first condition that is met triggers an update to the data.

- Deviation Threshold: A new aggregation round starts when a node identifies that the off-chain values deviate by more than the defined deviation threshold from the on-chain value. Individual nodes monitor one or more data providers for each feed.
- Heartbeat Threshold: A new aggregation round starts after a specified amount of time from the last update.

The [contract](https://sepolia.etherscan.io/address/0x5ACcd4dc94e91672E7aCd4D28D14a3A9ee131f7A#writeContract) at '0x5ACcd4dc94e91672E7aCd4D28D14a3A9ee131f7A' is an example of a very simple decentralized oracle implementation we created. Note that it is designed purely for an educational example and should not be used in production. There are many gas inefficiencies and obvious potential attack vectors, including a trivial Sybil attack.

The oracle works as follows:
- There are 3 public functions: `readValue()`, `reportValue(uint256)`, `processUpdate()`
- Anyone can call `readValue()` to get the current oracle value and the block number that the value was last updated
- Anyone can call `reportValue(uint256)` to report a value for the next round must pay `REQUIRED_PAYMENT` to do so
- Anyone can call `processUpdate()` after `UPDATE_INTERVAL` blocks has passed since the last update
    - The caller receives 10% of all payments in the round as an incentive
    - The median value of all reported values is chosen as the latest oracle value
    - All accounts that reported the chosen value split the remaining 90% of funds
    
This is a simple example design of an oracle that incentivises people to report the "correct" value (current temprature in Nansha), as they will be rewarded for doing so.

In [ ]:
ORACLE_CONTRACT_ADDRESS = "0x5ACcd4dc94e91672E7aCd4D28D14a3A9ee131f7A"
ABI = '[{"inputs":[],"name":"REQUIRED_PAYMENT","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"UPDATE_INTERVAL","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"processUpdate","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"readValue","outputs":[{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"value","type":"uint256"}],"name":"reportValue","outputs":[],"stateMutability":"payable","type":"function"}]'

# Read the latest oracle value

Let's read the latest value from the oracle.

In [ ]:
contract = web3.eth.contract(address=ORACLE_CONTRACT_ADDRESS, abi=ABI)
value, block_number = contract.functions.readValue().call()

print(f"The current oracle value is {value}")
print(f"Value was last updated at block {block_number} ({web3.eth.block_number - block_number} blocks ago)")

## Reporting a value to the oracle

Now, let's a report a value to the oracle for the upcoming update. 

Remember, everyone who reports the median value during this round will split all the rewards, so it is in your interest to report a value as close as possible to what you think is the correct value. You can only make one report per round.

- Go to the [write contract page](https://sepolia.etherscan.io/address/0xA07b0598fb5eCB9B0C2B9045D1078d855f050a1c#writeContract) for the oracle on Etherscan.
- Make a transaction calling `reportValue()` by providing 0.01 Sepolia ETH along with the value you wish to report

Whenever at least `UPDATE_INTERVAL` blocks have passed since the last update, anyone can call `processUpdate()` to trigger the oracle update and rewards distribution. This example contract distributes Sepolia ETH as a reward, but you could easily imagine that the reward could be the protocol's token or something else similar instead.

Try being the first to call `processUpdate()` to get the caller reward for free. (HINT: calling `readValue()` also tells you the last block the value was updated at. The variable `UPDATE_INTERVAL` defines the minimum number of blocks between intervals.

## Reporting adversarial values

Oracles should be resilient to malicious input. For this part, we want approximately 1/3 of the class to report an adverserial value and hopefully demonstrate that our oracle still produces a reasonable value.

When instructed to do so:
- If TA assigned you as an adversary,
    - Call `reportValue()` and report an adverserial value
- Otherwise
    - Call `reportValue()` and report what you believe is the correct value

After the next update, call `readValue()` to see what value the oracle decided on. Hopefully, it seems reasonable!